<a href="https://colab.research.google.com/github/Hanzet22/TKJ-Dumps/blob/main/Projek_Silvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
#  🛡️ WebVuln Inspector v1.0
#  Produk TKJ - Karya: Silvi Riyani
#  Fungsi: Cek kerentanan dasar website
#  ⚠️ HANYA UNTUK EDUKASI & WEBSITE SENDIRI!
# ================================================

import requests
from urllib.parse import urljoin
from datetime import datetime

print("""
   ╔═══════════════════════════════════════════════════╗
   ║   🛡️  WebVuln Inspector v1.0                      ║
   ║   Cek Kerentanan Web Sederhana                    ║
   ╚═══════════════════════════════════════════════════╝
""")

print("⚠️ PERINGATAN:")
print("   Tool ini hanya untuk EDUKASI dan pengujian di WEBSITE SENDIRI.")
print("   Gunakan hanya dengan izin pemilik website!")
print("   Penulis tidak bertanggung jawab atas penyalahgunaan.\n")

target = input("🌐 Masukkan URL (contoh: https://example.com): ").strip()

if not target.startswith("http"):
    target = "https://" + target

print(f"\n🔍 Memindai: {target}")
print("="*50)

# ================================================
# 1. CEK STATUS HTTPS & REDIRECT
# ================================================
def check_https(target):
    print("\n🔒 [1] CEK HTTPS & REDIRECT")
    try:
        # Cek HTTP (80) redirect ke HTTPS
        http_url = target.replace("https://", "http://")
        r_http = requests.get(http_url, timeout=5, allow_redirects=True)
        final_url = r_http.url

        if final_url.startswith("https://"):
            print("   ✅ HTTP otomatis redirect ke HTTPS (Aman!)")
        else:
            print("   ⚠️ HTTP tidak redirect ke HTTPS!")
            print("      💡 Redireksi ke: " + final_url)
    except:
        print("   ❌ Gagal cek redirect HTTP")

    # Cek HTTPS langsung
    try:
        r_https = requests.get(target, timeout=5)
        print(f"   ✅ HTTPS aktif (Status: {r_https.status_code})")
        return r_https
    except:
        print("   ❌ HTTPS tidak aktif atau error!")
        return None

# ================================================
# 2. CEK SECURITY HEADERS
# ================================================
def check_security_headers(response):
    print("\n🛡️ [2] CEK SECURITY HEADERS")

    if response is None:
        print("   ❌ Tidak ada response, skip.")
        return

    headers = response.headers

    checks = {
        'Strict-Transport-Security (HSTS)': headers.get('Strict-Transport-Security'),
        'X-Frame-Options (Clickjacking)': headers.get('X-Frame-Options'),
        'X-Content-Type-Options (MIME Sniff)': headers.get('X-Content-Type-Options'),
        'Content-Security-Policy (CSP)': headers.get('Content-Security-Policy'),
        'Referrer-Policy': headers.get('Referrer-Policy'),
        'Server': headers.get('Server'),
        'X-Powered-By': headers.get('X-Powered-By'),
    }

    for name, value in checks.items():
        if value:
            print(f"   ✅ {name}: {value}")
        else:
            if 'Server' in name or 'X-Powered' in name:
                print(f"   ℹ️  {name}: {value or 'Tidak terdeteksi (baik)'}")
            else:
                print(f"   ❌ {name}: TIDAK ADA (Rentan!)")

# ================================================
# 3. CEK COOKIE SECURITY
# ================================================
def check_cookie_security(response):
    print("\n🍪 [3] CEK KEAMANAN COOKIE")

    if response is None:
        print("   ❌ Tidak ada response, skip.")
        return

    cookies = response.cookies
    if not cookies:
        print("   ℹ️  Tidak ada cookie yang diset.")
        return

    for cookie in cookies:
        print(f"   📡 Cookie: {cookie.name}")
        if cookie.secure:
            print(f"      ✅ Secure flag: YES")
        else:
            print(f"      ❌ Secure flag: NO (bisa dicuri lewat HTTP)")
        if cookie.has_nonstandard_attr('HttpOnly'):
            print(f"      ✅ HttpOnly flag: YES")
        else:
            print(f"      ❌ HttpOnly flag: NO (bisa diakses JavaScript)")

# ================================================
# 4. CEK FILE SENSITIF (robots.txt, .env, dll)
# ================================================
def check_sensitive_files(target):
    print("\n📂 [4] CEK FILE SENSITIF (deteksi status)")

    # Normalisasi base URL
    if not target.endswith('/'):
        target = target + '/'

    files = [
        'robots.txt',
        'sitemap.xml',
        'admin/',
        'login/',
        'phpmyadmin/',
        'backup.zip',
        'backup.sql',
        '.env',
        '.git/',
        'wp-admin/',
        'uploads/',
        'vendor/',
    ]

    found = []

    for file in files:
        url = urljoin(target, file)
        try:
            r = requests.get(url, timeout=3, allow_redirects=False)
            if r.status_code == 200:
                print(f"   ⚠️  {file} — STATUS: 200 (TERBUKA!)")
                found.append(file)
            elif r.status_code == 403:
                print(f"   ℹ️  {file} — STATUS: 403 (Terlindungi)")
            else:
                print(f"   ✅ {file} — STATUS: {r.status_code} (Aman)")
        except:
            print(f"   ❌ {file} — Tidak bisa diakses")

    if found:
        print(f"\n   ⚠️  TOTAL FILE SENSITIF TERBUKA: {len(found)}")
        print("      💡 Segera batasi akses ke file tersebut!")
    else:
        print("   ✅ Tidak ada file sensitif yang terbuka (baik)")

# ================================================
# 5. CEK SERVER / TECHNOLOGY STACK
# ================================================
def check_tech_stack(response):
    print("\n🖥️ [5] TEKNOLOGI YANG TERDETEKSI")

    if response is None:
        print("   ❌ Tidak ada response.")
        return

    server = response.headers.get('Server', 'N/A')
    x_powered = response.headers.get('X-Powered-By', 'N/A')

    print(f"   📡 Server: {server}")
    print(f"   ⚡ X-Powered-By: {x_powered}")

    if 'nginx' in server.lower():
        print("      ℹ️  Nginx detected. Pastikan versi terbaru.")
    elif 'apache' in server.lower():
        print("      ℹ️  Apache detected. Pastikan mod_security aktif.")
    elif 'cloudflare' in server.lower():
        print("      ✅ Cloudflare detected (proteksi DDoS).")

# ================================================
# 6. REKOMENDASI AKHIR
# ================================================
def show_final_recommendations(response, found_sensitive):
    print("\n" + "="*50)
    print("📋 REKOMENDASI KEAMANAN")
    print("="*50)

    risks = 0

    if response:
        headers = response.headers
        if not headers.get('Strict-Transport-Security'):
            print("   ❌ HSTS tidak ada → Aktifkan HSTS!")
            risks += 1
        if not headers.get('X-Frame-Options'):
            print("   ❌ X-Frame-Options tidak ada → Tambahkan 'SAMEORIGIN'!")
            risks += 1
        if not headers.get('X-Content-Type-Options'):
            print("   ❌ X-Content-Type-Options tidak ada → Tambahkan 'nosniff'!")
            risks += 1
        if not headers.get('Content-Security-Policy'):
            print("   ⚠️ CSP tidak ada → Tambahkan CSP untuk keamanan ekstra.")

    if found_sensitive:
        print(f"   ❌ {len(found_sensitive)} file sensitif terbuka → Batasi akses!")
        risks += len(found_sensitive)

    if risks == 0:
        print("   ✅ WEBSITE TERLIHAT AMAN! Pertahankan.")
        print("      💡 Rutin lakukan update dan scanning.")
    else:
        print(f"\n   ⚠️  Ditemukan {risks} celah/risiko. Segera perbaiki!")

    print(f"\n💡 Saran Umum:")
    print("   1. Selalu gunakan HTTPS dengan sertifikat valid.")
    print("   2. Sembunyikan versi server (Server / X-Powered-By).")
    print("   3. Aktifkan Web Application Firewall (WAF).")
    print("   4. Rutin backup & update CMS/plugin.")

    print(f"\n📋 Waktu scan: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("✨ Scan selesai!")

# ================================================
# EKSEKUSI
# ================================================
response = check_https(target)
check_security_headers(response)
check_cookie_security(response)
sensitive_files = check_sensitive_files(target)
check_tech_stack(response)
show_final_recommendations(response, sensitive_files)